# Validation protocol and baselines

This protocol is fixed for the whole project, so any difference between two models comes from the model and not from the split:
- target: log1p(SalePrice)
- 5-fold KFold, shuffled, random_state=42, defined once in src/cv.py
- score: RMSE on log price, reported as mean ± std over the 5 folds
- Kaggle's test set is never used for tuning

The baselines are the numbers every real model has to beat.

In [1]:
# Make src/ importable from notebooks/ and reload edited modules automatically
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer

from src.cv import RAW_DIR, CV, SCORING, evaluate, show_results
from src.baseline import GroupMedianRegressor
from src.features import make_linear_preprocessor

train = pd.read_csv(RAW_DIR / "train.csv")
X = train.drop(columns=["Id", "SalePrice"])
y = np.log1p(train["SalePrice"])

Everything comes from src/: the folds (CV), the scoring and evaluate(), which runs the CV, prints mean ± std and saves the row to results.csv. Re-running a model replaces its old row, so Run All doesn't create duplicates.

In [2]:
# The shared folds: the same split for every model in the project
for k, (tr_idx, va_idx) in enumerate(CV.split(X), start=1):
    print(f"fold {k}: train {len(tr_idx)} | valid {len(va_idx)} | "
          f"mean log price in valid {y.iloc[va_idx].mean():.3f} | std {y.iloc[va_idx].std():.3f}")

# The splitter is deterministic: splitting twice gives exactly the same folds
first = [va for _, va in CV.split(X)]
second = [va for _, va in CV.split(X)]
print("same folds every time:", all((a == b).all() for a, b in zip(first, second)))

fold 1: train 1168 | valid 292 | mean log price in valid 11.998 | std 0.433
fold 2: train 1168 | valid 292 | mean log price in valid 12.053 | std 0.395
fold 3: train 1168 | valid 292 | mean log price in valid 12.029 | std 0.376
fold 4: train 1168 | valid 292 | mean log price in valid 12.009 | std 0.421
fold 5: train 1168 | valid 292 | mean log price in valid 12.032 | std 0.369
same folds every time: True


Each fold has 292 houses for validation and 1,168 for training. The folds look alike: mean log price between 12.00 and 12.05, spread between 0.37 and 0.43. Fold 1 has the widest spread, probably because it holds house 692, the most expensive one.

Splitting twice gives exactly the same folds, so every model in the project is tested on the same houses.

In [3]:
# Baseline 1: predict the average log price for every house
b1 = DummyRegressor(strategy="mean")
s_b1 = evaluate(b1, X, y, "Dummy (mean)", "none", phase=3, n_features=0,
                notes="training-fold mean for every house")

# For comparison: the spread of the target itself
print("std of log price:", round(y.std(ddof=0), 4))

Dummy (mean) [none]: RMSE 0.3988 ± 0.0251 | folds [0.4332, 0.3958, 0.3754, 0.421, 0.3685]
std of log price: 0.3993


Predicting the average price for every house gives an RMSE of 0.399, almost exactly the standard deviation of log price (0.399). That makes sense: if you always predict the mean, your error is just the spread of the target. In percentage terms it's about 49% off. This is the floor.

In [4]:
# Baseline 2: the median log price of the house's neighborhood
b2 = GroupMedianRegressor(group_col="Neighborhood")
s_b2 = evaluate(b2, X, y, "Neighborhood median", "Neighborhood", phase=3, n_features=1,
                notes="location only")

Neighborhood median [Neighborhood]: RMSE 0.2690 ± 0.0094 | folds [0.2729, 0.2683, 0.2619, 0.2844, 0.2574]


Location alone brings the error down from 0.399 to 0.269, about a third less. It's also very stable across folds (std 0.009).

In [5]:
# Baseline 3: linear regression on one feature, OverallQual
b3 = Pipeline([
    ("select", ColumnTransformer([("qual", "passthrough", ["OverallQual"])])),
    ("lr", LinearRegression()),
])
s_b3 = evaluate(b3, X, y, "LinearRegression", "OverallQual", phase=3, n_features=1,
                notes="quality only")

LinearRegression [OverallQual]: RMSE 0.2308 ± 0.0069 | folds [0.2311, 0.2272, 0.2351, 0.2406, 0.2202]


One feature, OverallQual, does better than the neighborhood: 0.231. It's the most stable model so far (std 0.007).

In [6]:
# Baseline 4: linear regression on OverallQual + log(GrLivArea)
b4 = Pipeline([
    ("select", ColumnTransformer([
        ("qual", "passthrough", ["OverallQual"]),
        ("log_area", FunctionTransformer(np.log1p), ["GrLivArea"]),
    ])),
    ("lr", LinearRegression()),
])
s_b4 = evaluate(b4, X, y, "LinearRegression", "OverallQual + log(GrLivArea)", phase=3, n_features=2,
                notes="quality + size")

LinearRegression [OverallQual + log(GrLivArea)]: RMSE 0.1995 ± 0.0159 | folds [0.1992, 0.1883, 0.2228, 0.2096, 0.1774]


Adding log(GrLivArea) to OverallQual brings it to 0.1995, about 22% typical error, with only two features.

The std doubles (0.016), and fold 3 is again the worst (0.223). That's the fold with houses 524 and 1299: the model sees a huge living area and predicts a high price.

In [7]:
# Reference: the full Phase 2 pipeline with an untuned Ridge (all 280 features)
ridge = make_pipeline(make_linear_preprocessor(), Ridge(alpha=10))
s_ridge = evaluate(ridge, X, y, "Ridge (alpha=10, untuned)", "linear_full", params={"alpha": 10},
                   phase=3, n_features=280, notes="all features, no tuning; reference for Phases 4-5")

Ridge (alpha=10, untuned) [linear_full]: RMSE 0.1315 ± 0.0240 | folds [0.1278, 0.1231, 0.178, 0.1193, 0.1096]


All 280 features with an untuned Ridge: 0.1315, about 14% typical error. Fold 3 is again much worse (0.178 vs about 0.12 for the rest), which is why the std (0.024) is the highest in the table.

In [8]:
# The results table, best first, with the RMSE read as a typical % error
table = show_results()
table["typical_error_%"] = ((np.exp(table["cv_rmse_mean"]) - 1) * 100).round(1)
print(table[["model", "feature_set", "n_features", "cv_rmse_mean", "cv_rmse_std", "typical_error_%"]].to_string())

                       model                   feature_set  n_features  cv_rmse_mean  cv_rmse_std  typical_error_%
0  Ridge (alpha=10, untuned)                   linear_full         280       0.13155      0.02399             14.1
1           LinearRegression  OverallQual + log(GrLivArea)           2       0.19946      0.01586             22.1
2           LinearRegression                   OverallQual           1       0.23083      0.00692             26.0
3        Neighborhood median                  Neighborhood           1       0.26898      0.00937             30.9
4               Dummy (mean)                          none           0       0.39878      0.02513             49.0


Every step helps, but the gains get smaller:
- mean only: 0.399
- two features: 0.199, half the error gone
- 280 features: 0.132

The first two features cover about 75% of the improvement we have so far ((0.399 - 0.199) / (0.399 - 0.132)). The other 278 features add the remaining 25%. That's the diminishing returns point from the brief, and a good argument for starting simple.

typical_error_% is exp(RMSE) - 1: because we work in log price, RMSE reads roughly as a percentage error.

In [9]:
# Fold by fold: is Ridge better than baseline 4 on every fold, or just on average?
diff = s_b4 - s_ridge
print("baseline 4:", s_b4.round(4))
print("ridge:     ", s_ridge.round(4))
print("difference:", diff.round(4), "| Ridge better on", int((diff > 0).sum()), "of 5 folds")

baseline 4: [0.1992 0.1883 0.2228 0.2096 0.1774]
ridge:      [0.1278 0.1231 0.178  0.1193 0.1096]
difference: [0.0714 0.0653 0.0448 0.0903 0.0678] | Ridge better on 5 of 5 folds


Ridge beats baseline 4 on all 5 folds, by 0.045 to 0.090. These gaps are much bigger than the fold-to-fold noise, so the improvement is real, not luck of the split.

The smallest gain is in fold 3 (0.045). The outliers hurt the full model more than the 2-feature one, probably because Ridge combines several area features that all point to a very high price for those two houses.

In [10]:
# The same two models measured with three metrics
metrics = {"rmse": SCORING, "mae": "neg_mean_absolute_error", "r2": "r2"}
for name, model in [("baseline 4", b4), ("ridge", ridge)]:
    res = cross_validate(model, X, y, cv=CV, scoring=metrics, n_jobs=-1)
    print(f"{name:10s} RMSE {-res['test_rmse'].mean():.4f} | "
          f"MAE {-res['test_mae'].mean():.4f} | R² {res['test_r2'].mean():.3f}")

baseline 4 RMSE 0.1995 | MAE 0.1475 | R² 0.745
ridge      RMSE 0.1315 | MAE 0.0834 | R² 0.884


RMSE and MAE tell different stories. For Ridge, MAE is 0.083 (about 9% off for a typical house) but RMSE is 0.132. RMSE squares the errors, so a few big misses pull it up. The ratio RMSE / MAE is 1.58 for Ridge and 1.35 for baseline 4; with normally distributed errors it would be around 1.25. So Ridge is very good on most houses, and a handful of large errors (the outliers again) inflate its RMSE.

R² goes from 0.745 to 0.884. It's useful as "how much better than predicting the mean", but it doesn't say how big the errors are in price terms, so I report RMSE as the main number.

## Where I am after Phase 3

The protocol is fixed: same 5 folds, RMSE on log price, mean ± std, logged in results.csv.

The baselines to beat are 0.399 (mean), 0.269 (neighborhood), 0.231 (quality) and 0.1995 (quality + log area). An untuned Ridge with all features gets 0.1315 and beats baseline 4 on every fold.

The same two houses (524 and 1299) keep showing up: they make fold 3 the worst and they inflate RMSE compared with MAE.

Next, Phase 4: plain linear regression with full diagnostics (coefficients, p-values, assumptions, VIF), using statsmodels for inference and the same CV for scores.